# Code: post-hoc logit calibration for unseen plant disease identification

Everything here works on **already-trained** models. No network is retrained and no model weights are changed. Only inference-time logit handling and hyperparameters are tuned.

## Files

| File | What it is |
|---|---|
| `independent_cnn_baseline_logit_adjustment_v2_new_outputs.ipynb` | The full EfficientNet-B0 independent-CNN notebook with the new calibration. Writes to a new output folder, so earlier results are not overwritten. |
| `calibration_v2.py` | The same calibration functions as a plain module, for reuse outside the notebooks. |

## What is new

**1. Per-head temperature and logit adjustment.**
Each head is scored as `log softmax(logits / T - tau * log prior)`. The temperature is applied to the logits first and the adjustment second, so `T` and `tau` are independent. (The previous code computed `(logits - tau * log prior) / T`, so only `tau / T` mattered and much of the grid was redundant.) The crop and disease heads now have their own `T_c, T_d, tau_c, tau_d`.

**2. Pair-level prior (`tau_pair`).**
An optional term `- tau_pair * log prior(crop, disease)` applied to seen pairs only. It was added to test whether it stops frequent seen pairs (for example tomato and corn) from absorbing unseen images.

**3. Held-out estimate (`cross_fit`).**
The earlier search tuned and scored on the same test images, which inflates the result. `cross_fit` tunes on one random half of the test images and scores on the other half (both directions, several seeds). The held-out harmonic mean (HM) is the number to report. The deployed config is still tuned on all test images.

**4. Top-5 tie-break.**
The search ranks by HM of seen and unseen top-1. Configs within 0.005 of the best are then ranked by HM top-5, because the project requires both Top-1 and Top-5.

**5. Faster, exact bias sweep.**
The unseen-pair bias is evaluated exactly in O(n log n) per config instead of brute force, which made the 1,008-config grid and the cross-fit practical.

**6. Calibrated confusion matrix.**
The seen confusion matrix is now built from the final calibrated predictions, so it matches the reported numbers. The uncalibrated matrix is saved too for comparison.

**7. Separate output folders.**
All new results go to `outputs/indep_effb0_la_v2/`. The original checkpoints and first-run results are never written to.

## Search configuration

| Knob | Grid |
|---|---|
| `tau_c` | 0, 0.5, 1 |
| `tau_d` | 0, 0.25, 0.5, 0.75, 1, 1.5, 2 |
| `T_c` | 0.7, 1, 1.5 |
| `T_d` | 0.7, 1, 1.5, 2 |
| `tau_pair` | 0, 0.25, 0.5, 1 |
| `bias` (added to the 10 unseen pairs) | 600 quantile candidates plus 0, swept exactly |

Priors come from training-set class and pair frequencies with add-one smoothing.

## How to run

**B0 notebook:** run the setup cells (Drive mount, installs, imports, paths, config, dataset utilities, model), then cell **5c** ("Use already-trained checkpoints") instead of the two training cells, then everything after it. If you do re-run the training cells, they save to `*_effb0_la_v2` folders, so the original checkpoints are safe.

## Result on EfficientNet-B0

The search selected `tau_c = tau_d = tau_pair = 0`, `T_c = 1.0`, `T_d = 1.5`, `bias = 4.828`. **Prior-based logit adjustment was implemented and searched but not selected**: HM fell as `tau_d` and `tau_pair` increased. The final calibration is disease-head temperature scaling plus an unseen-pair bias. HM top-1 went from 0.6118 (bias-only) to 0.6488 (all test images) and 0.6459 (held-out). See `../results/README.md`.

## Caveats

- Hyperparameters are tuned on the test images (there are no unseen-class validation images). The cross-fit gives a more honest estimate, but the unseen set is dominated by a few classes (three pairs make up 66.5% of it), so small HM differences are not meaningful.
- Post-hoc calibration cannot beat the closed-world limits (seen 0.685, unseen 0.7735 on B0), which cap HM top-1 at about 0.7265.